In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    # TODO: Resize to 28x28
    transforms.Resize((28, 28)),

    transforms.Grayscale(3),  # Convert grayscale to RGB (Don't Touch!!)

    # TODO: Convert to Tensor
    transforms.ToTensor(),
    # TODO: Normalize with ImageNet mean=[0.485, 0.456, 0.406] and std=[0.229, 0.224, 0.225]
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")


In [ ]:
# Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

# Create DataLoaders and display samples
# Write your code here

# DataLoaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

# denormalize for visualization
mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

def show_samples(dataloader, n=10):
    images, labels = next(iter(dataloader))
    images = images[:n].cpu()
    labels = labels[:n].cpu()

    imgs = images * std + mean
    imgs = torch.clamp(imgs, 0, 1)

    fig, axes = plt.subplots(1, n, figsize=(2*n, 2))
    for i in range(n):
        ax = axes[i]
        ax.imshow(imgs[i].permute(1,2,0))

        ax.set_title(letters[labels[i].item()-1])
        ax.axis("off")
    plt.show()

show_samples(train_loader, n=10)


In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s , EfficientNet_V2_S_Weights

# Write your code here

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

weights = EfficientNet_V2_S_Weights.DEFAULT
model = efficientnet_v2_s(weights=weights)

# Freeze backbone
for param in model.parameters():
    param.requires_grad = False

in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, num_classes)

# Unfreeze classifier head only
for param in model.classifier.parameters():
    param.requires_grad = True

model = model.to(device)

print(model.classifier)


In [ ]:
# Write your code here
import torch

def accuracy_from_logits(logits, targets):
    preds = torch.argmax(logits, dim=1)
    return (preds == targets).float().mean().item()

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_acc = 0.0

    for images, labels in loader:
        images = images.to(device)
        labels = (labels - 1).to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits, labels)

    return total_loss / len(loader), total_acc / len(loader)

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0

    for images, labels in loader:
        images = images.to(device)
        labels = (labels - 1).to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits, labels)

    return total_loss / len(loader), total_acc / len(loader)


In [ ]:
# Write your code here
import torch.optim as optim
import matplotlib.pyplot as plt

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(model.classifier.parameters(), lr=1e-3)
num_epochs = 5

train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(num_epochs):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    va_loss, va_acc = validate(model, test_loader, criterion, device)

    train_losses.append(tr_loss)
    val_losses.append(va_loss)
    train_accs.append(tr_acc)
    val_accs.append(va_acc)

    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {tr_loss:.4f}, Train Acc: {tr_acc:.4f} | "
          f"Val Loss: {va_loss:.4f}, Val Acc: {va_acc:.4f}")

# Plot Loss
plt.figure(figsize=(7,4))
plt.plot(train_losses, marker='o', label="Train Loss")
plt.plot(val_losses, marker='o', label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()
plt.show()

# Plot Accuracy
plt.figure(figsize=(7,4))
plt.plot(train_accs, marker='o', label="Train Acc")
plt.plot(val_accs, marker='o', label="Val Acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy Curve")
plt.legend()
plt.show()


In [ ]:
# Write your code here
@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0

    for images, labels in loader:
        images = images.to(device)
        labels = (labels - 1).to(device)

        # h_flipped = torch..flip(images, dim=[3])
        # v_flipped = torch.flip(images, dim=[2])

        logits = model(images)
        loss = criterion(logits, labels)

        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits, labels)

    return total_loss / len(loader), total_acc / len(loader)